# t-SNE on Oil Production Data
## Machine learning for energy applications — Module 4, Exercise 2

**t-Distributed Stochastic Neighbor Embedding (t-SNE)** is a non-linear
dimensionality-reduction technique that excels at visualizing high-dimensional
data in 2D or 3D. Unlike PCA, t-SNE focuses on preserving *local* neighbourhood
structure rather than global variance.

In this exercise we:
1. Explore and standardize oil production data
2. Apply t-SNE and visualize the 2D embedding
3. Experiment with the perplexity hyperparameter and with two random seeds
4. Compare t-SNE with PCA on the same data
5. Score every map with **trustworthiness** at k = 10 neighbours

> **Exploratory analysis.** The scaler and every map are fitted on all 725 wells: the maps
> describe this table. scikit-learn's `TSNE` has no `transform()` for new rows, so a map cannot
> be reused for new wells, and an island in a map is not evidence of a physical population until
> it is checked in the original features.

---
## 0 — Imports and Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.manifold import TSNE, trustworthiness
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

plt.rcParams.update({
    "figure.figsize": (8, 5),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 12,
})

RANDOM_STATE = 42
print("Setup complete.")

---
## 1 — Load and Explore the Data

The dataset describes oil-production wells with geophysical and petrophysical measurements:
- **md (ft)** / **tvd (ft)** — measured depth and true vertical depth
- **well spacing** — distance to nearest well
- **shale footage** — thickness of shale interval
- **acoustic impedance** — product of density and velocity
- **log permeability** / **porosity** — reservoir quality indicators
- **poisson ratio** / **p-velocity** / **s-velocity** — elastic properties

In [ ]:
df = pd.read_csv("Oil_production.csv")
print(f"Shape: {df.shape}")
df.head(10)

In [ ]:
df.describe()

In [ ]:
df.info()

In [ ]:
# Correlation matrix
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(df.corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0,
            square=True, linewidths=0.5, ax=ax, annot_kws={"size": 8})
ax.set_title("Feature Correlation Matrix")
plt.tight_layout()
plt.show()

---
## 2 — Standardize Features

t-SNE, like PCA, benefits from standardized input because it computes distances
between points in the high-dimensional space. Without scaling, the depths and footages
(in feet) would dominate every distance. The scaler is fitted on all 725 rows because the maps
are exploratory.

In [ ]:
feature_cols = df.columns.tolist()
X = df[feature_cols].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"Features: {len(feature_cols)}")
print(f"Samples : {X_scaled.shape[0]}")
print(f"Means   : {X_scaled.mean(axis=0).round(6)}")

---
## 3 — Apply t-SNE with Default Perplexity

**Perplexity** is the key hyperparameter for t-SNE. It roughly corresponds to
the effective number of neighbours each point considers. Typical values range
from 5 to 50.

We also compute **trustworthiness** at k = 10: it penalizes points that are among the 10 nearest
neighbours in the map but were not neighbours in the original space (1 = no such intruders).

In [ ]:
tsne_default = TSNE(n_components=2, perplexity=30, random_state=RANDOM_STATE, max_iter=1000)
X_tsne_default = tsne_default.fit_transform(X_scaled)

print(f"t-SNE output shape    : {X_tsne_default.shape}")
print(f"KL divergence         : {tsne_default.kl_divergence_:.4f}")
print(f"Trustworthiness (k=10): {trustworthiness(X_scaled, X_tsne_default, n_neighbors=10):.3f}")

### 3.1 Visualize 2D Embedding (Coloured by Porosity)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
scatter = ax.scatter(X_tsne_default[:, 0], X_tsne_default[:, 1],
                     c=df["porosity"], cmap="viridis", alpha=0.7, s=30,
                     edgecolors="w", linewidth=0.3)
cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label("Porosity")
ax.set_xlabel("t-SNE Dimension 1")
ax.set_ylabel("t-SNE Dimension 2")
ax.set_title("t-SNE Embedding (perplexity=30, coloured by Porosity)")
plt.tight_layout()
plt.show()

---
## 4 — Experiment with Different Perplexity Values

The perplexity parameter has a significant effect on the resulting embedding:
- **Low perplexity** (5): focuses on very local structure, may produce many small clusters
- **High perplexity** (50): considers broader neighbourhoods, tends to produce more global structure

There is no single "best" perplexity — it depends on the data and the analysis goal.
Do not compare KL divergence across perplexities (each perplexity defines a different target);
compare the maps with trustworthiness at the same k instead.

In [ ]:
perplexities = [5, 15, 30, 50]
tsne_results = {}
trust_results = {}

for p in perplexities:
    tsne = TSNE(n_components=2, perplexity=p, random_state=RANDOM_STATE, max_iter=1000)
    tsne_results[p] = tsne.fit_transform(X_scaled)
    trust_results[p] = trustworthiness(X_scaled, tsne_results[p], n_neighbors=10)
    print(f"Perplexity={p:>2d}  KL divergence={tsne.kl_divergence_:.4f}  "
          f"trustworthiness(k=10)={trust_results[p]:.3f}")

### 4.1 Perplexity Comparison (2x2 Subplot)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

for ax, p in zip(axes.ravel(), perplexities):
    X_emb = tsne_results[p]
    sc = ax.scatter(X_emb[:, 0], X_emb[:, 1],
                    c=df["porosity"], cmap="viridis", alpha=0.7, s=25,
                    edgecolors="w", linewidth=0.3)
    ax.set_title(f"Perplexity = {p}, T(10) = {trust_results[p]:.3f}", fontsize=13)
    ax.set_xlabel("t-SNE Dim 1")
    ax.set_ylabel("t-SNE Dim 2")

fig.suptitle("t-SNE Embeddings — Effect of Perplexity", fontsize=15, y=1.01)
plt.tight_layout(rect=[0, 0, 0.89, 1])   # lay out the grid first, leaving room on the right
cbar_ax = fig.add_axes([0.91, 0.15, 0.02, 0.7])
fig.colorbar(sc, cax=cbar_ax, label="Porosity")
plt.show()

### 4.2 Two Random Seeds

With the default `init="pca"` the layout barely depends on `random_state`. To see how much a map
depends on its random start, use `init="random"` with two seeds, at a small and a large
perplexity. Compare what changes between the maps (arrangement and orientation of the groups)
with what does not (trustworthiness, local neighbourhoods).

In [ ]:
seed_maps = {}
for seed in (0, 1):
    for p in (5, 50):
        emb = TSNE(n_components=2, perplexity=p, init="random", random_state=seed,
                   max_iter=1000).fit_transform(X_scaled)
        seed_maps[(seed, p)] = (emb, trustworthiness(X_scaled, emb, n_neighbors=10))
        print(f"init='random'  seed={seed}  perplexity={p:>2d}  "
              f"trustworthiness(k=10)={seed_maps[(seed, p)][1]:.3f}")

fig, axes = plt.subplots(2, 2, figsize=(12, 11))
for ax, ((seed, p), (emb, t)) in zip(axes.ravel(), seed_maps.items()):
    ax.scatter(emb[:, 0], emb[:, 1], c=df["porosity"], cmap="viridis", alpha=0.7, s=15,
               edgecolors="w", linewidth=0.2)
    ax.set_title(f"seed = {seed}, perplexity = {p}, T(10) = {t:.3f}", fontsize=12)
    ax.set_xlabel("t-SNE Dim 1")
    ax.set_ylabel("t-SNE Dim 2")
fig.suptitle('Same data, init="random", different seeds (coloured by Porosity)', fontsize=14)
plt.tight_layout()
plt.show()

---
## 5 — Colouring by Different Features

It is helpful to explore the embedding coloured by different features to understand
what structure t-SNE has captured. These features are inputs of the map, so the colours show
which inputs vary smoothly across it; they are not independent evidence of groups.

In [ ]:
color_features = ["porosity", "log permeability", "acoustic impedance", "poisson ratio"]

fig, axes = plt.subplots(2, 2, figsize=(14, 12))
X_emb = tsne_results[30]  # use perplexity=30

for ax, feat in zip(axes.ravel(), color_features):
    sc = ax.scatter(X_emb[:, 0], X_emb[:, 1],
                    c=df[feat], cmap="viridis", alpha=0.7, s=25,
                    edgecolors="w", linewidth=0.3)
    plt.colorbar(sc, ax=ax, shrink=0.8)
    ax.set_title(f"Coloured by {feat}", fontsize=12)
    ax.set_xlabel("t-SNE Dim 1")
    ax.set_ylabel("t-SNE Dim 2")

plt.suptitle("t-SNE (perplexity=30) Coloured by Various Features", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

---
## 6 — Compare t-SNE with PCA

PCA and t-SNE optimize for different objectives:
- **PCA**: maximizes global variance; preserves large pairwise distances
- **t-SNE**: preserves local neighbourhoods; can distort global distances

Let us compare the two side by side.

In [ ]:
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_scaled)
trust_pca = trustworthiness(X_scaled, X_pca, n_neighbors=10)

print(f"PCA explained variance: PC1={pca.explained_variance_ratio_[0]:.3f}, "
      f"PC2={pca.explained_variance_ratio_[1]:.3f}, "
      f"Total={pca.explained_variance_ratio_.sum():.3f}")
print(f"Trustworthiness (k=10): PCA {trust_pca:.3f}, t-SNE (perplexity=30) {trust_results[30]:.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# PCA
sc1 = axes[0].scatter(X_pca[:, 0], X_pca[:, 1],
                      c=df["porosity"], cmap="viridis", alpha=0.7, s=25,
                      edgecolors="w", linewidth=0.3)
axes[0].set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)")
axes[0].set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)")
axes[0].set_title(f"PCA (T(10) = {trust_pca:.3f})")
plt.colorbar(sc1, ax=axes[0], shrink=0.8)

# t-SNE
sc2 = axes[1].scatter(tsne_results[30][:, 0], tsne_results[30][:, 1],
                      c=df["porosity"], cmap="viridis", alpha=0.7, s=25,
                      edgecolors="w", linewidth=0.3)
axes[1].set_xlabel("t-SNE Dim 1")
axes[1].set_ylabel("t-SNE Dim 2")
axes[1].set_title(f"t-SNE (perplexity=30, T(10) = {trust_results[30]:.3f})")
plt.colorbar(sc2, ax=axes[1], shrink=0.8)

plt.suptitle("PCA vs t-SNE — Coloured by Porosity", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
## 7 — Limitations and When to Use t-SNE

### Strengths
- Excellent for **visual exploration** of high-dimensional data
- Reveals clusters and local structure that linear methods like PCA miss
- Works well when the data lies on a non-linear manifold

### Limitations
- **Seed-dependent** — with `init="random"`, different seeds give different layouts; compare at least two
- **Perplexity-sensitive** — results change significantly with different perplexity values
- **No transform for new points** — scikit-learn's `TSNE` has no `transform()`; new wells cannot be placed on an existing map without refitting
- **Does not preserve global structure** — distances between clusters, and cluster sizes, in the plot may be meaningless
- **Computationally expensive** — O(n^2) for the exact algorithm; Barnes-Hut approximation helps
- **Not suitable for feature extraction** — the embedding dimensions have no interpretable meaning

### When to use t-SNE
- Exploring structure in high-dimensional data as a **visualization tool**
- When you suspect non-linear relationships that PCA would miss
- For a first look at possible groups before clustering; maps can exaggerate separation, so
  confirm any group in the original features

---
## Summary

In this exercise we applied t-SNE to oil production data and covered:

1. **Data preparation** — standardization is important for distance-based methods.
2. **t-SNE embedding** — projecting 10 features into 2D for visualization.
3. **Perplexity and seeds** — low perplexity captures local detail; high perplexity reveals broader patterns; with `init="random"` the layout changes with the seed.
4. **Trustworthiness** — a neighbourhood score at a stated k, comparable across settings, seeds and methods.
5. **Feature colouring** — exploring which features vary across the embedding.
6. **PCA comparison** — PCA preserves global variance while t-SNE preserves local neighbourhoods.
7. **Limitations** — t-SNE is a visualization tool, not a general-purpose dimensionality reduction method.

**Key takeaway**: t-SNE is a powerful tool for exploring high-dimensional oil and gas
data, but its results should be interpreted carefully. Always try multiple perplexity
values and at least two seeds, report the settings and the trustworthiness with every map,
and use it alongside PCA for a complete picture.